In [117]:
import pandas as pd
import numpy as np

In [144]:
import pandas as pd

# 1. Configuration
raw_file = 'raw_data'  # Update this to your file name
chunk_size = 500000

print(f"--- Analyzing {raw_file} ---")

# 2. Get Column Names (Info) without loading any data
# We read 0 rows just to get the header
headers = pd.read_csv(raw_file, nrows=0).columns.tolist()
print(f"\n1. Columns Found in Raw Data:\n{headers}")

# 3. Get Unique Years and Total Row Count using Chunking
# We use 'usecols' so Python only looks at the YEAR column, saving huge amounts of RAM
unique_years = set()
total_rows = 0

print(f"\n2. Scanning file for row count and unique years...")

try:
    for chunk in pd.read_csv(raw_file, usecols=['YEAR'], chunksize=chunk_size):
        total_rows += len(chunk)
        unique_years.update(chunk['YEAR'].unique())
        print(f"   Processed {total_rows:,} rows...")

    print("\n" + "="*40)
    print("FINAL RAW DATA REPORT")
    print("="*40)
    print(f"Total Records (N):  {total_rows:,}")
    print(f"Unique Survey Years: {sorted(list(unique_years))}")
    print(f"Number of Columns:   {len(headers)}")
    print("="*40)

except Exception as e:
    print(f"\nError: {e}")
    print("Tip: Ensure the column name is exactly 'YEAR' (case-sensitive).")

--- Analyzing raw_data ---

1. Columns Found in Raw Data:
['YEAR', 'SAMPLE', 'SERIAL', 'CBSERIAL', 'HHWT', 'CLUSTER', 'STATEFIP', 'STRATA', 'GQ', 'OWNERSHP', 'OWNERSHPD', 'MORTGAGE', 'RENT', 'PERNUM', 'PERWT', 'BPL', 'BPLD', 'YRIMMIG', 'EDUC', 'EDUCD', 'INCTOT']

2. Scanning file for row count and unique years...
   Processed 500,000 rows...
   Processed 1,000,000 rows...
   Processed 1,500,000 rows...
   Processed 2,000,000 rows...
   Processed 2,500,000 rows...
   Processed 3,000,000 rows...
   Processed 3,500,000 rows...
   Processed 4,000,000 rows...
   Processed 4,500,000 rows...
   Processed 5,000,000 rows...
   Processed 5,500,000 rows...
   Processed 6,000,000 rows...
   Processed 6,500,000 rows...
   Processed 7,000,000 rows...
   Processed 7,500,000 rows...
   Processed 8,000,000 rows...
   Processed 8,500,000 rows...
   Processed 9,000,000 rows...
   Processed 9,500,000 rows...
   Processed 10,000,000 rows...
   Processed 10,500,000 rows...
   Processed 11,000,000 rows...
  

In [1]:
import pandas as pd
import numpy as np

# 1. SETUP - Ensure 'raw_data' is the correct name of your 3.4GB file
input_file = 'raw_data' 
output_file = 'HCWCI_Master_Dataset.csv'

# Mappings for the paper
country_map = {
    20000: 'Mexico', 26010: 'Dominican Republic', 50000: 'China',
    51500: 'Philippines', 51800: 'Vietnam', 52100: 'India',
    52110: 'Bangladesh', 54200: 'Turkey', 60031: 'Nigeria', 60045: 'Kenya'
}

state_map = {
    1: 'Alabama', 2: 'Alaska', 4: 'Arizona', 5: 'Arkansas', 6: 'California', 8: 'Colorado', 
    9: 'Connecticut', 10: 'Delaware', 11: 'DC', 12: 'Florida', 13: 'Georgia', 15: 'Hawaii', 
    16: 'Idaho', 17: 'Illinois', 18: 'Indiana', 19: 'Iowa', 20: 'Kansas', 21: 'Kentucky', 
    22: 'Louisiana', 23: 'Maine', 24: 'Maryland', 25: 'Massachusetts', 26: 'Michigan', 
    27: 'Minnesota', 28: 'Mississippi', 29: 'Missouri', 30: 'Montana', 31: 'Nebraska', 
    32: 'Nevada', 33: 'New Hampshire', 34: 'New Jersey', 35: 'New Mexico', 36: 'New York', 
    37: 'North Carolina', 38: 'North Dakota', 39: 'Ohio', 40: 'Oklahoma', 41: 'Oregon', 
    42: 'Pennsylvania', 44: 'Rhode Island', 45: 'South Carolina', 46: 'South Dakota', 
    47: 'Tennessee', 48: 'Texas', 49: 'Utah', 50: 'Vermont', 51: 'Virginia', 53: 'Washington', 
    54: 'West Virginia', 55: 'Wisconsin', 56: 'Wyoming'
}

def get_educ_group(e):
    if e <= 2: return "1_Low (No/Primary)"
    elif e <= 5: return "2_Some High School"
    elif e == 6: return "3_HS Graduate"
    elif e <= 8: return "4_Some College"
    else: return "5_College+ (Bachelor or Higher)"

# 2. CHUNKING ENGINE
print(f"Reading {input_file} in chunks...")
cols = ['YEAR', 'PERWT', 'STATEFIP', 'BPLD', 'YRIMMIG', 'OWNERSHP', 'MORTGAGE', 'RENT', 'EDUC', 'INCTOT']

processed_list = []
chunk_count = 0

# Read in pieces of 100,000 rows to save RAM
for chunk in pd.read_csv(input_file, usecols=cols, chunksize=100000):
    
    # --- NEW: FILTER OUT ABNORMAL INCOME CODES (9999999) ---
    # This removes N/A and non-response codes before aggregation
    chunk = chunk[chunk['INCTOT'] < 9999998].copy()
    
    # Filter for target countries and actual immigrants
    chunk['Country'] = chunk['BPLD'].map(country_map)
    chunk = chunk.dropna(subset=['Country']).copy()
    chunk = chunk[chunk['YRIMMIG'] > 0].copy() 
    
    if not chunk.empty:
        # Create variables for the HC-WCI model
        chunk['State_Name'] = chunk['STATEFIP'].map(state_map)
        chunk['Education_Group'] = chunk['EDUC'].apply(get_educ_group)
        chunk['Years_in_US'] = chunk['YEAR'] - chunk['YRIMMIG']
        
        # Cap at 30 years for the paper timeline
        chunk = chunk[chunk['Years_in_US'] <= 30].copy()
        
        # Calculate Weighted Values (Critical for Census Accuracy)
        # Mortgage filter: Owners (1) with a mortgage (2+)
        chunk['Has_Mortgage'] = np.where((chunk['OWNERSHP'] == 1) & (chunk['MORTGAGE'] > 1), 1, 0)
        chunk['W_Mortgages'] = chunk['Has_Mortgage'] * chunk['PERWT']
        chunk['W_Income'] = chunk['INCTOT'] * chunk['PERWT']
        chunk['W_Rent'] = chunk['RENT'] * chunk['PERWT']
        
        # Group immediately to shrink data before storing
        summary = chunk.groupby(['Country', 'State_Name', 'Years_in_US', 'Education_Group']).agg(
            Total_People=('PERWT', 'sum'),
            Sum_Mortgages=('W_Mortgages', 'sum'),
            Sum_Income=('W_Income', 'sum'),
            Sum_Rent=('W_Rent', 'sum')
        ).reset_index()
        
        processed_list.append(summary)
    
    chunk_count += 1
    if chunk_count % 10 == 0:
        print(f"Processed {chunk_count * 100000} rows...")

# 3. FINAL AGGREGATION
print("Consolidating final dataset...")
final_df = pd.concat(processed_list).groupby(['Country', 'State_Name', 'Years_in_US', 'Education_Group']).sum().reset_index()

# 4. STATE RENT BENCHMARKING (The Rg variable)
# Only use records with positive rent to find the median market rate
state_market = final_df[final_df['Sum_Rent'] > 0].groupby('State_Name').apply(
    lambda x: x['Sum_Rent'].sum() / x['Total_People'].sum()
).reset_index()
state_market.columns = ['State_Name', 'State_Median_Rent']

final_df = final_df.merge(state_market, on='State_Name', how='left')

# 5. CALCULATE FINAL PAPER METRICS
# These are the columns you will drag into Tableau
final_df['Mortgage_Rate_Pct'] = (final_df['Sum_Mortgages'] / final_df['Total_People']) * 100
final_df['Avg_Income'] = final_df['Sum_Income'] / final_df['Total_People']
final_df['Avg_Rent_Paid'] = final_df['Sum_Rent'] / final_df['Total_People']
final_df['Rent_Performance_Ratio'] = final_df['Avg_Rent_Paid'] / final_df['State_Median_Rent']

# 6. SAVE
final_df.to_csv(output_file, index=False)
print(f"DONE! {output_file} created. 9999999 codes removed. Capped at 30 years.")

Reading raw_data in chunks...
Processed 1000000 rows...
Processed 2000000 rows...
Processed 3000000 rows...
Processed 4000000 rows...
Processed 5000000 rows...
Processed 6000000 rows...
Processed 7000000 rows...
Processed 8000000 rows...
Processed 9000000 rows...
Processed 10000000 rows...
Processed 11000000 rows...
Processed 12000000 rows...
Processed 13000000 rows...
Processed 14000000 rows...
Processed 15000000 rows...
Processed 16000000 rows...
Processed 17000000 rows...
Processed 18000000 rows...
Processed 19000000 rows...
Processed 20000000 rows...
Processed 21000000 rows...
Processed 22000000 rows...
Processed 23000000 rows...
Processed 24000000 rows...
Processed 25000000 rows...
Processed 26000000 rows...
Processed 27000000 rows...
Processed 28000000 rows...
Processed 29000000 rows...
Processed 30000000 rows...
Processed 31000000 rows...
Processed 32000000 rows...
Processed 33000000 rows...
Processed 34000000 rows...
Processed 35000000 rows...
Consolidating final dataset...


C:\Users\Amirh\AppData\Local\Temp\ipykernel_44092\3516715622.py:90: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  state_market = final_df[final_df['Sum_Rent'] > 0].groupby('State_Name').apply(


DONE! HCWCI_Master_Dataset.csv created. 9999999 codes removed. Capped at 30 years.


In [3]:
print(f"Number of states analyzed: {len(state_map)}")

Number of states analyzed: 51
